In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append("..")

In [ ]:
import torch
import torch.fft

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors

import numpy as np  

import time

from collections import OrderedDict

import shapely
import shapely.affinity
from shapely.ops import clip_by_rect
from skfem import Basis, ElementTriP0
from skfem.io.meshio import from_meshio

from femwell.maxwell.waveguide import compute_modes
from femwell.mesh import mesh_from_OrderedDict

from architectures import PINO2D
from generators import RibWaveguideDataset
from utils import full_vectorial_fd

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [ ]:
model = PINO2D(in_channels=5, out_channels=2, width=64, modes1=32, modes2=32, depth=4).to(device)
model = torch.nn.DataParallel(model)
if compile and torch.__version__ >= "2.0.0":
    model = torch.compile(model)

In [ ]:
checkpoint = torch.load(f'model_checkpoints/rib_checkpoint_2.pth', weights_only=False)

In [ ]:
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
dataset = RibWaveguideDataset(n_dataset=4, stochastic=False, device=device, dx=1/20, dy=1/20)

In [ ]:
def compute_Ez_from_divergence(Ex, Ey, xy, eps, beta):
    B, _, Nx, Ny = Ex.shape

    eps_Ex = eps * Ex
    eps_Ey = eps * Ey

    x = xy[:, 0:1, :, :]
    dx_e = x[:, :, 2:, :] - x[:, :, 1:-1, :]
    dx_w = x[:, :, 1:-1, :] - x[:, :, :-2, :]

    depsEx_dx = torch.zeros_like(Ex)
    depsEx_dx[:, :, 1:-1, :] = (eps_Ex[:, :, 2:, :] - eps_Ex[:, :, :-2, :]) / (dx_e + dx_w)

    y = xy[:, 1:2, :, :]
    dy_n = y[:, :, :, 2:] - y[:, :, :, 1:-1]
    dy_s = y[:, :, :, 1:-1] - y[:, :, :, :-2]

    depsEy_dy = torch.zeros_like(Ey)
    depsEy_dy[:, :, :, 1:-1] = (eps_Ey[:, :, :, 2:] - eps_Ey[:, :, :, :-2]) / (dy_n + dy_s)

    depsEx_dx[:, :, 0, :] = depsEx_dx[:, :, 1, :]
    depsEx_dx[:, :, -1, :] = depsEx_dx[:, :, -2, :]
    depsEy_dy[:, :, :, 0] = depsEy_dy[:, :, :, 1]
    depsEy_dy[:, :, :, -1] = depsEy_dy[:, :, :, -2]

    divergence_transverse = depsEx_dx + depsEy_dy
    
    Ez = -1j * (divergence_transverse / (beta * eps))
    return Ez

In [ ]:
specs = [
    {
    "core_width": 3.0,
    "core_height": 1.95,
    "rib_height": 1.05,
    "wavelength": 1.55,
    "n_core": 3.44,
    "n_sub": 2.95,
    "n_clad": 1.95,
    "TE" : 0.0
}
]
xyr_list, features_list = dataset.generate_from_specs(specs)

In [ ]:
fundamental_colors = ["#1E90FF", "#FFFFFF", "#FF69B4"] 
custom_cmap = mcolors.LinearSegmentedColormap.from_list("black_center", fundamental_colors)

errors = []
for sample_idx, (xyr, features) in enumerate(zip(xyr_list, features_list)):

    xyr = xyr.unsqueeze(0).to(device)
    features = features.unsqueeze(0).to(device)

    k_0 = features[0, 0].item()
    core_length_x = features[0, 1].item()
    core_length_y = features[0, 2].item()
    rib_height = features[0, 3].item()

    xy = xyr[0, :2].permute(1, 2, 0)
    n_map = xyr[0, 2]

    with torch.no_grad():
        start_time = time.time()
        E_pred, beta_nn = model(xyr)
        end_time = time.time()

    print(f'FNO took {round(end_time - start_time, 5)} seconds')
    selected_component = features[:, -1].to(torch.int64)
    batch_idx = torch.arange(E_pred.shape[0], device=device)

    E = torch.zeros_like(E_pred)
    E[batch_idx, selected_component] = E_pred[:, 0]
    E[batch_idx, 1 - selected_component] = E_pred[:, 1]

    n2_max = features[:, -2]
    n2_min = features[:, -3]

    beta_sq = (n2_max - beta_nn * (n2_max - n2_min)).item()
    neff_nn = np.sqrt(beta_sq)

    with torch.no_grad():
        Ez_nn = compute_Ez_from_divergence(
            E[:, 0:1],
            E[:, 1:2],
            xyr[:, :2],
            xyr[:, 2:3],
            (beta_sq**0.5 * k_0)
        )
        # Ez_nn = compute_Ez_from_divergence(
        #     E[:, 0:1],
        #     E[:, 1:2],
        #     xyr[:, 2:3],
        #     1/20,
        #     1/20,
        #     beta_sq * k_0**2
        # )
        Ez_nn = Ez_nn.imag[0, 0].cpu()

    start_time = time.time()

    eig_vals, modes_y, modes_x, _ = full_vectorial_fd(
        xy=xy,
        eps_map=n_map,
        dx=dataset.dx,
        dy=dataset.dy,
        k0=k_0,
        num_modes=2
    )

    end_time = time.time()

    print(f'FD took {round(end_time - start_time, 5)} seconds')

    eig_vals = torch.from_numpy(eig_vals)
    modes_x = torch.from_numpy(modes_x)
    modes_y = torch.from_numpy(modes_y)

    errors.append(torch.min(torch.abs(eig_vals - neff_nn)))

    Ex_nn = E[0, 0].detach().cpu()
    Ey_nn = E[0, 1].detach().cpu()

    nn_ratio = (
        torch.linalg.norm(Ex_nn) /
        torch.linalg.norm(Ey_nn)
    ).item()

    fd_ratios = [
        (
            torch.linalg.norm(modes_x[j]) /
            torch.linalg.norm(modes_y[j])
        ).item()
        for j in range(2)
    ]

    overlaps = []

    for j in range(2):

        fdx = modes_x[j].real
        fdy = modes_y[j].real

        ov = (
            torch.abs(torch.sum(Ex_nn * fdx)) +
            torch.abs(torch.sum(Ey_nn * fdy))
        )

        overlaps.append(ov.item())
        
    overlaps_t = []
    Ex_nnt = Ey_nn.detach().clone()
    Ey_nnt = Ex_nn.detach().clone()


    for j in range(2):

        fdx = modes_x[j].real
        fdy = modes_y[j].real

        ov = (
            torch.abs(torch.sum(Ex_nnt * fdx)) +
            torch.abs(torch.sum(Ey_nnt * fdy))
        )

        overlaps_t.append(ov.item())
        
    if np.max(overlaps) > np.max(overlaps_t):
        best_fd_idx = np.argmax(overlaps)
    else:
        best_fd_idx = np.argmax(overlaps_t)
        Ex_nn = Ex_nnt
        Ey_nn = Ey_nnt
    beta_sq_fd = eig_vals[best_fd_idx].item()

    with torch.no_grad():
        Ez_fd = compute_Ez_from_divergence(
            modes_x[best_fd_idx][None, None],
            modes_y[best_fd_idx][None, None],
            xyr[:, :2].cpu(),
            xyr[:, 2:3].cpu(),
            (beta_sq_fd * k_0)
        )
        # Ez_fd = compute_Ez_from_divergence(
        #     modes_x[best_fd_idx][None, None],
        #     modes_y[best_fd_idx][None, None],
        #     xyr[:, 2:3].cpu(),
        #     1/20,
        #     1/20,
        #     (beta_sq_fd * k_0)**2
        # )
        Ez_fd = Ez_fd.imag[0, 0].cpu()
        # print('*'*10, torch.max(Ez_fd), '*'*10)

    Ex_fd = modes_x[best_fd_idx].real
    Ey_fd = modes_y[best_fd_idx].real

    dominant_is_Ex = nn_ratio > 1.0

    def normalize_field(field_x, field_y):
        max_val = max(
            torch.max(torch.abs(field_x)),
            torch.max(torch.abs(field_y))
        )

        return (
            (field_x / max_val) * torch.sgn(torch.sum(field_x)),
            (field_y / max_val) * torch.sgn(torch.sum(field_y)),
            max_val
        )

    Ex_nn, Ey_nn, max_val_nn = normalize_field(Ex_nn, Ey_nn)
    Ex_fd, Ey_fd, max_val_fd = normalize_field(Ex_fd, Ey_fd)

    Ez_nn = Ez_nn / max_val_nn * torch.sgn(torch.sum(Ez_nn))
    Ez_fd = Ez_fd / max_val_fd * torch.sgn(torch.sum(Ez_fd))

    X = xy[:, :, 0].cpu()
    Y = xy[:, :, 1].cpu()

    if dominant_is_Ex:
        dom_nn, dom_fd = Ex_nn, Ex_fd
        min_nn, min_fd = Ey_nn, Ey_fd
        dom_label = r"$E_x$"
        min_label = r"$E_y$"
        mode_label = "TE-like"
    else:
        dom_nn, dom_fd = Ey_nn, Ey_fd
        min_nn, min_fd = Ex_nn, Ex_fd
        dom_label = r"$E_y$"
        min_label = r"$E_x$"
        mode_label = "TM-like"

    def sym_clim(*fields):
        v = max(torch.max(torch.abs(f)).item() for f in fields)
        return -v, v

    row_clims = [
        sym_clim(dom_nn, dom_fd),
        sym_clim(min_nn, min_fd),
        sym_clim(Ez_nn, Ez_fd),
    ]

    def slice_ylim(*fields):
        lo = min(f.min().item() for f in fields)
        hi = max(f.max().item() for f in fields)
        pad = 0.1 * max(abs(lo), abs(hi))
        return lo - pad, hi + pad

    row_ylims = [
        slice_ylim(dom_nn, dom_fd),
        slice_ylim(min_nn, min_fd),
        slice_ylim(Ez_nn, Ez_fd),
    ]

    fig = plt.figure(figsize=(18, 8))

    wavelength = round((2 * np.pi / k_0), 3)

    fig.suptitle(
        f"{mode_label} Mode   |   λ = {wavelength} μm   |   n_eff = {neff_nn:.5f}",
        fontsize=18,
        fontweight='bold'
    )

    gs = GridSpec(
        2,
        3,
        figure=fig,
        width_ratios=[1, 1, 1],
        hspace=0.28,
        wspace=0.28
    )

    dx = dataset.dx
    dy = dataset.dy

    x_edge = int((core_length_x / 2) / dx)
    y_edge = int((core_length_y) / dy)

    center_x = X.shape[0] // 2
    center_y = X.shape[1] // 2

    edge_x = center_x + int(0.75 * x_edge)
    edge_y = center_y + int(0.75 * y_edge)

    edge_x = np.clip(edge_x, 0, X.shape[0] - 1)
    edge_y = np.clip(edge_y, 0, X.shape[1] - 1)

    if dominant_is_Ex:
        ez_slice_x = edge_x
        ez_slice_y = center_y + int(0.5 * y_edge)
    else:
        ez_slice_x = center_x
        ez_slice_y = edge_y

    slice_positions = {
        "dom": (center_x, center_y + int(0.5 * y_edge)),
        "min": (edge_x + int(0.15 * x_edge), edge_y - int(0.5 * y_edge)),
        "ez":  (ez_slice_x, ez_slice_y),
    }

    def draw_waveguide(ax):
        ax.hlines(core_length_y, -core_length_x/2, core_length_x/2, colors='black', linewidth=1.2, linestyle='-')        
        ax.hlines(0, -4.9, -core_length_x/2, colors='black', linewidth=1.2, linestyle='-')
        ax.hlines(0, core_length_x/2, 4.9, colors='black', linewidth=1.2, linestyle='-')
        ax.axhline(-rib_height, color='black', linewidth=1.5, linestyle='-')

        ax.vlines([core_length_x/2, -core_length_x/2], 0, core_length_y, colors='black', linewidth=1.2, linestyle='-')

    def draw_slice_lines(ax, slice_key):

        ix, iy = slice_positions[slice_key]

        ax.axvline(
            X[ix, iy],
            color='black',
            linestyle='--',
            linewidth=1.5,
            alpha=0.95
        )

        ax.axhline(
            Y[ix, iy],
            color='black',
            linestyle='--',
            linewidth=1.5,
            alpha=0.95
        )
    def plot_2d_heatmap(ax, Z, title, vmin, vmax, slice_key):
        norm = mcolors.SymLogNorm(linthresh=0.35 * vmax, vmin=vmin, vmax=vmax, base=10)

        ax.pcolormesh(
            X,
            Y,
            Z,
            cmap=custom_cmap,
            shading='auto',
            norm = norm
            # vmin=vmin,
            # vmax=vmax,
        )

        draw_waveguide(ax)
        draw_slice_lines(ax, slice_key)

        ax.set_title(title, fontsize=15)
        # ax.set_xlim([-3.5, 3.5])
        # ax.set_ylim([-3.5, 3.5])
        ax.tick_params(axis='x', labelsize=12) 
        ax.tick_params(axis='y', labelsize=12) 
        ax.set_xlabel('Y (μm)', fontsize=13)
        ax.set_ylabel('x (μm)', fontsize=13)
        # ax.set_xticks([])
        # ax.set_yticks([])

    def plot_1d_slices(ax, Z_nn, Z_fd, ylim, slice_key, component_name, polarization):

        ix, iy = slice_positions[slice_key]
        Z_fd = Z_fd.detach().clone()
        if torch.sum(Z_nn * Z_fd) < 0:
            Z_fd *= -1

        x_coords = X[:, iy]
        y_coords = Y[ix, :]

        ri_x = n_map[:, iy].cpu()
        ri_y = n_map[ix, :].cpu()

        ri_x = ri_x / ri_x.max()
        ri_y = ri_y / ri_y.max()

        scaled_x = ''
        scaled_y = ''

        if polarization == 'x':
            ri_y = torch.ones_like(ri_y)
            scaled_x += '$\epsilon$'    

        elif polarization == 'y':
            ri_x = torch.ones_like(ri_x)
            scaled_y += '$\epsilon$'

        elif polarization == 'z':
            ri_x = torch.ones_like(ri_x)
            ri_y = torch.ones_like(ri_y)


        ax.plot(
            x_coords[::2],
            (ri_x * Z_nn[:, iy])[::2],
            linewidth=2.5,
            color='black',
            # label= scaled_x + component_name + ' (h-slice,PINO)'
            label= scaled_x + component_name + r'$(y)$' + ' (PINO)'
        )

        ax.plot(
            x_coords[::2],
            (ri_x * Z_fd[:, iy])[::2],
            '--',
            color='red',
            linewidth=2.5,
            label= scaled_x + component_name + r'$(y)$' + ' (FD)'
            # label= scaled_x + component_name + ' (h-slice, FD)'
        )

        ax.plot(
            y_coords,
            ri_y * Z_nn[ix, :],
            linewidth=2.5,
            color='#FF69B4',
            alpha=0.9,
            # label=scaled_y + component_name + ' (v-slice,PINO)'
            label=scaled_y + component_name + r'$(x)$' +  ' (PINO)'
        )

        ax.plot(
            y_coords,
            ri_y * Z_fd[ix, :],
            '--',
            color='#1E90FF',
            linewidth=2.5,
            alpha=0.9,
            # label= scaled_y + component_name + ' (v-slice,FD)'
            label=scaled_y + component_name + r'$(x)$' +  ' (FD)'
        )

        ax.axhline(0, color='k', linestyle=':', linewidth=0.9)

        # ax.set_xlim([-3.5, 3.5])
        # ax.set_ylim(ylim)
        ax.tick_params(axis='x', labelsize=12) 

        ax.grid(alpha=0.3)

        ax.legend(frameon=False)

    heatmap_data = [
        (dom_nn, f"{dom_label}", row_clims[0], "dom"),
        (min_nn, f"{min_label}", row_clims[1], "min"),
        (Ez_nn,  r"$E_z$", row_clims[2], "ez"),
    ]

    slice_data = [
        (dom_nn, dom_fd, row_ylims[0], "dom", f"{dom_label}"),
        (min_nn, min_fd, row_ylims[1], "min", f"{min_label}"),
        (Ez_nn,  Ez_fd,  row_ylims[2], "ez",  r"$E_z$"),
    ]


    for col, (field, title, clim, slice_key) in enumerate(heatmap_data):

        ax = fig.add_subplot(gs[0, col])

        plot_2d_heatmap(
            ax,
            field,
            title,
            clim[0],
            clim[1],
            slice_key
        )

    polarizations = [
        'x' if dominant_is_Ex else 'y',
        'y' if dominant_is_Ex else 'x',
        'z'
    ]

    for col, (nn_field, fd_field, ylim, slice_key, component_name) in enumerate(slice_data):

        ax = fig.add_subplot(gs[1, col])

        plot_1d_slices(
            ax,
            nn_field,
            fd_field,
            ylim,
            slice_key,
            component_name,
            polarizations[col]
        )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

print(
    'average error is',
    round(sum(errors).item(), 6) / len(errors)
)

In [ ]:
def get_frequency_variations(xyr, features, step_size):
    k_0 = features[0, 0].item()
    steps = [-2*step_size, -step_size, 0, step_size, 2*step_size]
    variations = []
    for i, step in enumerate(steps):
        variations.append(xyr.clone())
        variations[-1][:, -1, :, :] = (k_0 + step) ** 2
    return variations

In [ ]:
def calculate_dispersion_fd(
    xyr_base,
    features_base,
    step_size_lambda,
    model,
    device
):
    n_eff_values = []
    k0_values = []

    # ---------------------------------------------------
    # PINO reference mode for overlap tracking
    # ---------------------------------------------------
    with torch.no_grad():
        E_out, beta_nn = model(xyr_base.to(device))

    selected_component = features_base[:, -1].to(torch.int64)
    batch_idx = torch.arange(E_out.shape[0], device=device)

    E = torch.zeros_like(E_out)

    E[batch_idx, selected_component] = E_out[:, 0]
    E[batch_idx, 1 - selected_component] = E_out[:, 1]

    Ex_ref = E[:, 0:1].cpu()
    Ey_ref = E[:, 1:2].cpu()

    # ---------------------------------------------------
    # Frequency sweep
    # ---------------------------------------------------
    for xyr in get_frequency_variations(
        xyr_base,
        features_base,
        step_size=step_size_lambda
    ):

        xy = xyr[0, :2].permute(1, 2, 0)
        n_map = xyr[0, 2]

        k0 = torch.sqrt(xyr[0, -1, 0, 0]).item()

        eig_vals, modes_y, modes_x, _ = full_vectorial_fd(
            xy=xy,
            eps_map=n_map,
            dx=dataset.dx,
            dy=dataset.dy,
            k0=k0,
            num_modes=2
        )

        overlaps = []

        for j in range(2):

            Ex_fd = modes_x[j].real
            Ey_fd = modes_y[j].real

            ov = (
                torch.abs(torch.sum(Ex_ref * Ex_fd)) +
                torch.abs(torch.sum(Ey_ref * Ey_fd))
            )

            overlaps.append(ov.item())
            
        overlaps_t = []
        Ex_ref_t = Ey_ref.detach().clone()
        Ey_ref_t = Ex_ref.detach().clone()

        for j in range(2):

            Ex_fd = modes_x[j].real
            Ey_fd = modes_y[j].real

            ov = (
                torch.abs(torch.sum(Ex_ref_t * Ex_fd)) +
                torch.abs(torch.sum(Ey_ref_t * Ey_fd))
            )

            overlaps_t.append(ov.item())
        
        if np.max(overlaps) > np.max(overlaps_t):
            best_fd_idx = np.argmax(overlaps)
        else:
            best_fd_idx = np.argmax(overlaps_t)
            
        if eig_vals[best_fd_idx] > eig_vals[1-best_fd_idx]:
            which_mode = "higher"
        else:
            which_mode = "lower"

        n_eff_values.append(eig_vals[best_fd_idx].item())
        k0_values.append(k0)

    n_eff_values = np.array(n_eff_values)
    k0_values = np.array(k0_values)

    h = k0_values[1] - k0_values[0]

    d2n_k2 = (
        -n_eff_values[0]
        + 16 * n_eff_values[1]
        - 30 * n_eff_values[2]
        + 16 * n_eff_values[3]
        - n_eff_values[4]
    ) / (12 * h**2)
    
    dn_dk = (
    -n_eff_values[4]
    + 8 * n_eff_values[3]
    - 8 * n_eff_values[1]
    + n_eff_values[0]
    ) / (12 * h)

    return 2 * dn_dk + k0_values[2] * d2n_k2, which_mode


def calculate_dispersion_pino(
    xyr_base,
    features_base,
    model
):

    xyr = xyr_base.clone()

    k0 = torch.tensor(
        [features_base[0, 0].item()],
        requires_grad=True
    )
    xyr[:, -1] = k0**2

    eta_nn = model(xyr)[1].squeeze()

    n_eff_squared = (
        features_base[0, -2]
        - eta_nn * (
            features_base[:, -2]
            - features_base[:, -3]
        )
    )

    n_eff = torch.sqrt(n_eff_squared)

    dn_dk = torch.autograd.grad(
        n_eff.sum(),
        k0,
        create_graph=True
    )[0]

    d2n_dk2 = torch.autograd.grad(
        dn_dk.sum(),
        k0,
        create_graph=True
    )[0]
    
    return 2 * dn_dk.detach() + k0.detach() * d2n_dk2.detach()

In [ ]:
def evaluate_fem(param_dict, return_modes=False):
    core_width = param_dict["core_width"]
    core_height = param_dict["core_height"]
    rib_height = param_dict["rib_height"]
    wavelength = param_dict["wavelength"]
    core = param_dict['n_core']
    sub = param_dict['n_sub']
    clad = param_dict['n_clad']
    dxy = param_dict.get("dxy", 0.1)

    xmin, xmax = -5, 5
    ymin, ymax = -5, 5

    core_interface = shapely.Polygon((
        (-5, -rib_height), (5, -rib_height), (5, 0),
        (core_width/2, 0), (core_width/2, core_height),
        (-core_width/2, core_height), (-core_width/2, 0),
        (-5, 0), (-5, -rib_height)
    ))
    env = shapely.geometry.box(xmin, ymin, xmax, ymax)
    
    polygons = OrderedDict(
        core=core_interface,
        box=clip_by_rect(env, -np.inf, -np.inf, np.inf, 0),
        clad=clip_by_rect(env, -np.inf, 0, np.inf, np.inf),
    )

    resolutions = dict(core={"resolution": dxy, "distance": 0.5})

    mesh = from_meshio(mesh_from_OrderedDict(polygons, resolutions, default_resolution_max=1, default_resolution_min=dxy))
    basis0 = Basis(mesh, ElementTriP0())
    epsilon = basis0.zeros()
    
    for subdomain, n in {"core": core, "box": sub, "clad": clad}.items():
        epsilon[basis0.get_dofs(elements=subdomain)] = n**2
        
    # Compute 2 modes to allow tracking selection
    modes = compute_modes(basis0, epsilon, wavelength=wavelength, num_modes=2, order=2)
    
    if return_modes:
        return modes
    return [(mode.n_eff).item() for mode in modes]

def calculate_dispersion_fem(param_dict, step_size, which_mode='higher'):
    
    lambda_base = param_dict["wavelength"]
    k0_base = 2 * np.pi / lambda_base
    steps = [-2 * step_size, -step_size, 0, step_size, 2 * step_size]
    
    n_eff_values = []
    k0_values = []
    local_params = param_dict.copy()
    
    for step in steps:
        k0_step = k0_base + step
        local_params["wavelength"] = 2 * np.pi / k0_step
        
        current_neffs = evaluate_fem(local_params, return_modes=False)
        
        if which_mode == 'higher':
            current_neff = np.max([n_eff.real for n_eff in current_neffs])
        else:
            current_neff = np.min([n_eff.real for n_eff in current_neffs])
            
        n_eff_values.append(current_neff)
        k0_values.append(k0_step)
        
    h = k0_values[1] - k0_values[0]
    
    d2n_k2 = (-n_eff_values[0] + 16 * n_eff_values[1] - 30 * n_eff_values[2] + 16 * n_eff_values[3] - n_eff_values[4]) / (12 * h**2)
    dn_dk = (-n_eff_values[4] + 8 * n_eff_values[3] - 8 * n_eff_values[1] + n_eff_values[0]) / (12 * h)
    
    return 2 * dn_dk + k0_values[2] * d2n_k2

In [ ]:
def evaluate_dispersion_sweep(
    model,
    sweep_results,
    param_dict_list,
    device,
    step_sizes,
    models=['pino', 'fd', 'fem']
):
    """
    Evaluates PINO, FDM, and FEM dispersion terms simultaneously over a parameter sweep.
    Accepts an array/list of step_sizes for the numerical derivative approaches.
    """
    dispersion_pino = []
    # Nested lists to handle multiple step sizes for FD and FEM
    dispersion_fd = [[] for _ in range(len(step_sizes))]
    dispersion_fem = [[] for _ in range(len(step_sizes))]

    model.eval()

    # Loop over the sweep index (e.g. tracking index across changing n_core)
    for idx in range(len(sweep_results[0])):
        xyr, features = sweep_results[0][idx], sweep_results[1][idx]
        xyr = xyr.unsqueeze(0).to(device)
        features = features.unsqueeze(0).to(device)
        
        # Current geometry/material dictionary for the FEM solver
        current_param_dict = param_dict_list[idx]

        # ---------------- 1. PINO Dispersion ----------------
        if 'pino' in models:
            # PINO is exact/analytical and does not require a discrete step size
            pino_val = calculate_dispersion_pino(xyr, features, model)
            # Extracted scalar value handling tensor detaches
            dispersion_pino.append(pino_val.item() if hasattr(pino_val, 'item') else pino_val)

        # ---------------- 2. Multi-Step Numerical Dispersion (FD & FEM) ----------------
        for i, step in enumerate(step_sizes):
            
            # --- FDM ---
            if 'fd' in models:
                fd_val, which_mode = calculate_dispersion_fd(
                    xyr_base=xyr, 
                    features_base=features, 
                    step_size_lambda=step, 
                    model=model, 
                    device=device
                )
                dispersion_fd[i].append(fd_val)

            # --- FEM ---
            if 'fem' in models:
                now = time.time()
                fem_val = calculate_dispersion_fem(
                    param_dict=current_param_dict, 
                    step_size=step,
                    which_mode=which_mode if 'fd' in models else 'higher'
                )
                dispersion_fem[i].append(fem_val)

        print(f'Completed dispersion evaluation for sample {idx+1}/{len(sweep_results[0])}')

    return {
        "dispersion_pino": dispersion_pino,
        "dispersion_fd": dispersion_fd,
        "dispersion_fem": dispersion_fem
    }

def evaluate_n_eff_sweep(
    model,
    sweep_results,
    param_dict_list,
    device,
    FD=True,
    FEM=True
):
    eigenvalues_pino = []
    eigenvalues_fd = []
    eigenvalues_fem = []

    model.eval()

    for idx in range(len(sweep_results[0])):
        xyr, features = sweep_results[0][idx], sweep_results[1][idx]
        xyr = xyr.unsqueeze(0).to(device)
        features = features.unsqueeze(0).to(device)

        dx = dataset.dx
        dy = dataset.dy
        k0 = features[0, 0].item()
        n2_max = features[0, -2]
        n2_min = features[0, -3]

        with torch.no_grad():
            E_out, beta_nn = model(xyr)

        beta_sq = (n2_max - beta_nn * (n2_max - n2_min))
        beta_pino = torch.sqrt(beta_sq).item()

        selected_component = features[:, -1].to(torch.int64)
        batch_idx = torch.arange(E_out.shape[0], device=device)

        E = torch.zeros_like(E_out)

        E[batch_idx, selected_component] = E_out[:, 0]
        E[batch_idx, 1 - selected_component] = E_out[:, 1]

        Ex_nn = E[:, 0:1].cpu()
        Ey_nn = E[:, 1:2].cpu()

        # ---------------- FD ----------------
        if FD:
            xy = xyr[0, :2].permute(1, 2, 0)
            n_map = xyr[0, 2]
            eig_vals, modes_y, modes_x, _ = full_vectorial_fd(
                xy=xy,
                eps_map=n_map,
                dx=dx,
                dy=dy,
                k0=k0,
                num_modes=2
            )

            overlaps = []
        
            for j in range(2):
        
                fdx = modes_x[j].real
                fdy = modes_y[j].real
        
                ov = (
                    torch.abs(torch.sum(Ex_nn * fdx)) +
                    torch.abs(torch.sum(Ey_nn * fdy))
                )
        
                overlaps.append(ov.item())
        
            Ex_nn_copy  = Ey_nn.detach().clone()
            Ey_nn_copy = Ex_nn.detach().clone()
        
            overlaps_t = []
            for j in range(2):
        
                fdx = modes_x[j].real
                fdy = modes_y[j].real
        
                ov = (
                    torch.abs(torch.sum(Ex_nn_copy * fdx)) +
                    torch.abs(torch.sum(Ey_nn_copy * fdy))
                )
        
                overlaps_t.append(ov.item())
        
            if np.max(overlaps) > np.max(overlaps_t):
                best_fd_idx = np.argmax(overlaps)
            else:
                best_fd_idx = np.argmax(overlaps_t)
            neff_fd = eig_vals[best_fd_idx]
            eigenvalues_fd.append(neff_fd.item())

        # ---------------- FEM ----------------
        if FEM:
            now = time.time()
            fem_vals = evaluate_fem(param_dict_list[idx])
            best_fem_idx = np.argmin(
                [abs(beta_pino - r) for r in fem_vals]
            )
            eigenvalues_fem.append(fem_vals[best_fem_idx].real)
            print('fem took' , round(time.time() - now, 5), 'seconds')
        eigenvalues_pino.append(beta_pino)
        print(f"Completed evaluation for sample {idx+1}/{len(sweep_results[0])}")

    return {
        "eigenvalues_pino": eigenvalues_pino,
        "eigenvalues_fd": eigenvalues_fd,
        "eigenvalues_fem": eigenvalues_fem
    }


In [ ]:
dataset = RibWaveguideDataset(device=device, dx=1/40, dy=1/40)

In [ ]:
from copy import deepcopy

def build_grid_aligned_values(vmin, vmax, dx, mode="center"):
    """
    Generate values that are exactly representable on the grid.
    """
    if mode == "center":
        k_min = int(np.ceil(vmin / dx - 0.5))
        k_max = int(np.floor(vmax / dx - 0.5))
        values = (np.arange(k_min, k_max + 1) + 0.5) * dx

    elif mode == "edge":
        k_min = int(np.ceil(vmin / dx))
        k_max = int(np.floor(vmax / dx))
        values = (np.arange(k_min, k_max + 1)) * dx

    else:
        raise ValueError

    return values

# ---------------- Base template ----------------
dx = 1/5
base_params = {
    "core_width": (int(3.0/dx))*dx,
    "core_height": (int(1.0/dx))*dx,
    "rib_height": (int(1.0/dx))*dx,
    "n_core": 3.48,
    "n_sub": 2.5,
    "n_clad": 1.5,
    "TE": False,
    "wavelength": 1.55
}

# ---------------- Unified sweep configuration ----------------
param_sweeps = [
    dict(
        name=r"$w_{rib}$",
        key="core_width",
        values=build_grid_aligned_values(2.05, 4.95, dx=dx, mode='edge'),
    ),
    dict(
        name=r"$h_{rib}$",
        key="core_height",
        values=build_grid_aligned_values(0.15, 2.95, dx=dx, mode='edge'),
    ),
    dict(
        name=r"$h_{slab}$",
        key="rib_height",
        values=build_grid_aligned_values(0.15, 2.95, dx=dx, mode='edge'),
    ),
    dict(
        name=r"$n_{co}$",
        key="n_core",
        values=np.linspace(2.55, 4.45, 15),
    ),
    dict(
        name=r"$n_{sub}$",
        key="n_sub",
        values=np.linspace(2.05, 3.35, 15),
    ),
    dict(
        name=r"$n_{clad}$",
        key="n_clad",
        values=np.linspace(1.05, 2.35, 15),
    )
]

# ---------------- Generator ----------------
def generate_param_dicts(sweep, base_params):
    vals = sweep["values"]
    key = sweep["key"]

    param_dict_list = []
    for v in vals:
        d = deepcopy(base_params)
        d[key] = float(v)
        param_dict_list.append(d)

    return sweep["name"], vals, param_dict_list


In [ ]:
plot_colors = ['#1E90FF', 'black']
fem_color = '#7B68EE'   # medium slate blue
j = 0
sweep_results_list = []
for sweep in param_sweeps:
    parameter_name, parameter_range, param_dict_list = \
        generate_param_dicts(sweep, base_params)

    sweep_samples = dataset.generate_from_specs(param_dict_list)

    sweep_results = evaluate_n_eff_sweep(
        model,
        sweep_samples,
        param_dict_list,
        device,
        FD=True,
        FEM=True
    )

    sweep_results_list.append(sweep_results)
    torch.save(sweep_results_list, f'sweep_results_list.pth')
    # sweep_results = torch.load(f'sweep_results.pth')[j]
    # sweep_results = sweep_results_list[j]
    plt.plot(
        parameter_range, sweep_results['eigenvalues_fd'],
        label= '$FD$',
        linewidth=2.5,
        color = plot_colors[0],
        alpha=0.9
    )
    plt.scatter(
        parameter_range, np.array(sweep_results['eigenvalues_pino']),
        label='PINO',
        color = '#FF69B4',
        s=80,
        facecolors='none',        # hollow center
        edgecolors = '#FF69B4',
        alpha=0.9,
        zorder=3
    )

    plt.scatter(
        parameter_range,
        sweep_results['eigenvalues_fem'],
        label='FEM',
        color=fem_color,
        marker='s',          # square
        s=85,
        facecolors='none',
        edgecolors=fem_color,
        linewidth=1.5,
        alpha=0.9,
        zorder=2
    )
    # 
    # plt.scatter(
    #     parameter_range,
    #     sweep_results['eigenvalues_fem'],
    #     label='$FEM$',
    #     marker='*',
    #     color='green',
    #     s=80
    # )
    plt.xlabel(parameter_name, fontsize=16)
    plt.ylabel(r"$n_{eff}$", fontsize=16)

    if j==0:
        plt.axvline(
            x=4.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
        plt.annotate('in-distribution', (3.0, 3.452), fontsize=12)
        plt.annotate('out-of-distribution', (4.1, 3.452), fontsize=12)

    elif j in [1, 2]:
        plt.axvline(
            x=2.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
        plt.annotate('in-distribution', (1.1, 3.44), fontsize=12)
        plt.annotate('out-of-distribution', (2.1, 3.44), fontsize=12)

    elif j==3:
        plt.axvline(
            x=3.5,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
        plt.annotate('in-distribution', (2.75, 3.5), fontsize=12)
        plt.annotate('out-of-distribution', (3.75, 3.5), fontsize=12)

    elif j==4:
        plt.axvline(
            x=3.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
        plt.annotate('in-distribution', (2.25, 3.455), fontsize=12)
        plt.annotate('  out-of-\ndistribution', (3.1, 3.452), fontsize=12)

    if j==5:
        plt.axvline(
            x=2.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
        plt.annotate('in-distribution', (1.25, 3.4545), fontsize=12)
        plt.annotate('  out-of-\ndistribution', (2.1, 3.4535), fontsize=12)
        plt.ylim([3.453, 3.455])
    plt.legend(frameon=False, fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    # plt.savefig('rib_sweep_new_' + parameter_name[1:-1] + '.png')
    plt.show()
    j+=1

In [ ]:
# dataset = RibWaveguideDataset(device=device, dx=1/20, dy=1/20) # to make things a bit faster for FD

In [ ]:
plot_colors = ['#1E90FF', 'black']
fem_color = '#7B68EE'   # medium slate blue
j = 0
sweep_dispersion_results_list = []
for sweep in param_sweeps:
    parameter_name, parameter_range, param_dict_list = \
        generate_param_dicts(sweep, base_params)

    sweep_samples = dataset.generate_from_specs(param_dict_list)
    
    step_sizes = [0.05] if j!=3 else [0.05, 0.025]
    sweep_results = evaluate_dispersion_sweep(
        model._orig_mod if hasattr(model, '_orig_mod') else model,
        sweep_samples,
        param_dict_list,
        device,
        step_sizes= step_sizes,
        models=['pino', 'fd', 'fem']
    )    

    sweep_dispersion_results_list.append(sweep_results)
    torch.save(sweep_dispersion_results_list, f'sweep_dispersion_results_list.pth')

    if j==3:
        k_range = [0, 1]
        labels = ['$FD \ (dk_{0}=$' + str(0.025) + '$)$', '$FD \ (dk_{0}=$' + str(0.01) + '$)$'] 
        fem_labels = ['$FEM \ (dk_{0}=$' + str(0.025) + '$)$', '$FEM \ (dk_{0}=$' + str(0.01) + '$)$']
    else:
        k_range=[0]
        labels = ['$FD$']
        fem_labels = ['$FEM$']
        
    for k in k_range:
        plt.plot(
            parameter_range, sweep_results['dispersion_fd'][k],
            label=labels[k] ,
            linewidth=2.5,
            color = plot_colors[k],
            alpha=0.9
        )
    for k in k_range:
        plt.scatter(
            parameter_range, np.array(sweep_results['dispersion_fem'][k]),
            label=fem_labels[k],
            marker='s',          # square
            color = ['#FF69B4', 'black'][k],
            s=80,
            facecolors='none',        # hollow center
            edgecolors = ['#FF69B4', 'black'][k],
            alpha=0.9,
            zorder=3
        )
    
    plt.scatter(
        parameter_range, np.array(sweep_results['dispersion_pino']),
        label='PINO',
        color = '#FF69B4',
        s=80,
        facecolors='none',        # hollow center
        edgecolors = '#FF69B4',
        alpha=0.9,
        zorder=3
    )

    plt.xlabel(parameter_name, fontsize=16)
    plt.ylabel(r"$\beta_2$", fontsize=16)
    # 
    if j==0:
        plt.axvline(
            x=4.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
    #     plt.annotate('in-distribution', (3.0, 3.452), fontsize=12)
    #     plt.annotate('out-of-distribution', (4.1, 3.452), fontsize=12)
    # 
    elif j in [1, 2]:
        plt.axvline(
            x=2.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
    #     plt.annotate('in-distribution', (1.1, 3.44), fontsize=12)
    #     plt.annotate('out-of-distribution', (2.1, 3.44), fontsize=12)
    # 
    elif j==3:
        plt.axvline(
            x=3.5,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
    #     plt.annotate('in-distribution', (2.75, 3.5), fontsize=12)
    #     plt.annotate('out-of-distribution', (3.75, 3.5), fontsize=12)
    # 
    elif j==4:
        plt.axvline(
            x=3.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
    #     plt.annotate('in-distribution', (2.25, 3.455), fontsize=12)
    #     plt.annotate('  out-of-\ndistribution', (3.1, 3.4540), fontsize=12)
    # 
    if j==5:
        plt.axvline(
            x=2.0,
            linestyle='--',
            linewidth=3,
            alpha=0.7,
        )
    #     plt.annotate('in-distribution', (1.25, 3.4545), fontsize=12)
    #     plt.annotate('  out-of-\ndistribution', (2.1, 3.4535), fontsize=12)
    #     plt.ylim([3.453, 3.455])
    plt.legend(frameon=False, fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.4)
    # plt.savefig('rib_dispersion_sweep_new_' + parameter_name[1:-1] + '.png')
    plt.tight_layout()
    plt.show()
    j+=1